# RANA AI — one-prompt film studio (open source, free Kaggle GPU)

**What this does:** installs **WanGP** (open source, low-VRAM AI video studio with the *Deepy Prime* film agent), opens a private web link, and you type **one brief**. Deepy writes the shot plan, keeps the cast consistent, generates the video shots (Wan 2.2 / LTX-2 open models), voice (OmniVoice — supports Telugu, Hindi, English), music and assembles the film. The last cell adds RANA AI branding, Telugu subtitles and the end card and exports a 1080×1920 MP4.

**Before you press *Run All*:**
1. *Settings → Accelerator →* **GPU T4 x2** · *Internet →* **On** (needs a phone-verified Kaggle account).
2. *Add-ons → Secrets →* add `WANGP_AUTH_PASSWORD` = a long private passphrase (protects your web link).
3. Kaggle gives about 30 free GPU hours a week, max 12 h per session. First run downloads models (~25–40 GB) and takes 30–60 min.

Everything here is open source. No paid API is used unless *you* choose one inside WanGP.

In [ ]:
# 1) GPU + driver check, pick a matching PyTorch build
import subprocess, re
smi = subprocess.run(['nvidia-smi'], capture_output=True, text=True).stdout
print(smi[:900])
drv = float(re.search(r'Driver Version: (\d+\.\d+)', smi).group(1)) if 'Driver Version' in smi else 0
if drv >= 580:   TORCH = 'torch==2.10.0 torchvision==0.25.0 torchaudio==2.10.0 --index-url https://download.pytorch.org/whl/cu130'
elif drv >= 570: TORCH = 'torch==2.7.1 torchvision==0.22.1 torchaudio==2.7.1 --index-url https://download.pytorch.org/whl/cu128'
else:            TORCH = 'torch==2.7.1 torchvision==0.22.1 torchaudio==2.7.1 --index-url https://download.pytorch.org/whl/cu126'
assert drv > 0, 'No GPU: set Accelerator to GPU T4 x2'
print('driver', drv, '->', TORCH.split('--')[0])

In [ ]:
# 2) Get WanGP (pinned, verified commit) into /tmp (more disk than /kaggle/working)
WANGP_COMMIT = '5b5ce9f'   # verified 16 Oct 2026; change to 'main' to update
!rm -rf /tmp/Wan2GP && git clone -q https://github.com/deepbeepmeep/Wan2GP.git /tmp/Wan2GP
!cd /tmp/Wan2GP && git checkout -q {WANGP_COMMIT} 2>/dev/null; git -C /tmp/Wan2GP log -1 --format='%h %cd %s'

In [ ]:
# 3) Install (T4 = Turing: Triton 3.2 + SageAttention 1)  ~10 min
!pip -q install {TORCH}
!cd /tmp/Wan2GP && pip -q install -r requirements.txt
!pip -q install "triton==3.2.0" "sageattention==1.0.6" || echo 'sage optional - falling back to sdpa'
!apt-get -qq install -y fonts-noto-core fonts-noto-ui-core >/dev/null 2>&1; fc-list | grep -ci telugu

In [ ]:
# 4) RANA finishing tool (branding, subtitles, end card, 1080x1920 export)
RANA_FILM_RAW = 'https://raw.githubusercontent.com/sricharanmunagala39-del/Rana.AI/work-film/film'
!mkdir -p /kaggle/working/rana && cd /kaggle/working/rana && wget -q -O finish.py {RANA_FILM_RAW}/finish.py && wget -q -O BRIEF_TEMPLATE.md {RANA_FILM_RAW}/BRIEF_TEMPLATE.md && ls -la

In [ ]:
# 5) Start the studio with a private link (Gradio share).  Open the printed *.gradio.live link,
#    enter your WANGP_AUTH_PASSWORD, then go to  <link>/deepy/  and paste your one brief.
# First time only: Configuration > Prompt Enhancer / Deepy > Deepy Prime + 'Qwen3.8VL Uncensored 9B' (GGUF Q4).
import os, subprocess, time
from kaggle_secrets import UserSecretsClient
os.environ['WANGP_AUTH_PASSWORD'] = UserSecretsClient().get_secret('WANGP_AUTH_PASSWORD')
os.makedirs('/kaggle/working/outputs', exist_ok=True)
attn = 'sage' if subprocess.run(['python','-c','import sageattention'],capture_output=True).returncode==0 else 'sdpa'
log = open('/kaggle/working/wangp.log','w')
proc = subprocess.Popen(['python','wgp.py','--listen','--share','--auth','--profile','4','--attention',attn],
                        cwd='/tmp/Wan2GP', stdout=log, stderr=subprocess.STDOUT)
for _ in range(240):
    time.sleep(5); txt = open('/kaggle/working/wangp.log').read()
    links = [w for w in txt.split() if 'gradio.live' in w]
    if links: print('OPEN:', links[-1], '\nDeepy film agent:', links[-1].rstrip('/') + '/deepy/'); break
else: print(open('/kaggle/working/wangp.log').read()[-3000:])

### Your one prompt
Paste the brief from `BRIEF_TEMPLATE.md` into Deepy. Ask it to save the final film and an `.srt` of the dialogue. When it says the film is done, run the next cell.

In [ ]:
# 6) Finish: newest film from WanGP -> RANA-branded Reels MP4
import glob, os, subprocess, json
vids = sorted(glob.glob('/tmp/Wan2GP/outputs/**/*.mp4', recursive=True) + glob.glob('/kaggle/working/outputs/**/*.mp4', recursive=True), key=os.path.getmtime)
assert vids, 'No video yet - let Deepy finish first'
raw = vids[-1]; srt = os.path.splitext(raw)[0] + '.srt'
cmd = ['python','/kaggle/working/rana/finish.py','--video',raw,'--out','/kaggle/working/RANA_final.mp4']
if os.path.exists(srt): cmd += ['--srt', srt]
print(subprocess.run(cmd, capture_output=True, text=True).stdout)
from IPython.display import FileLink; FileLink('RANA_final.mp4')